In [4]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

CHUNK_SIZE = 100_000
SAMPLE_SIZE = 10_000
RANDOM_SEED = 42
from pathlib import Path
import pandas as pd

ROOT = Path("student_resource")

# Training files
train_s1_path = ROOT / "dataset" / "train" / "train_source1.tsv"
train_s2_path = ROOT / "dataset" / "train" / "train_source2.tsv"
train_s3_path = ROOT / "dataset" / "train" / "train_source3.tsv"
ground_truth_path = ROOT / "dataset" / "train" / "train_ground_truth.tsv"

# Test files
test_s1_path = ROOT / "dataset" / "test" / "test_source1.tsv"
test_s2_path = ROOT / "dataset" / "test" / "test_source2.tsv"
test_s3_path = ROOT / "dataset" / "test" / "test_source3.tsv"

print("Paths configured.")


Paths configured.


In [5]:
def scan_source_file(path, sample_size=10_000, chunksize=100_000, seed=42):

    rng = np.random.default_rng(seed)

    total_rows = 0

    missing = {
        "entity_id": 0,
        "business_name": 0,
        "business_address": 0,
        "country": 0
    }

    country_counts = {}

    # Statistics for string lengths
    stats = {
        "name": {
            "count": 0,
            "sum": 0.0,
            "sum_sq": 0.0,
            "min": np.inf,
            "max": -np.inf
        },
        "address": {
            "count": 0,
            "sum": 0.0,
            "sum_sq": 0.0,
            "min": np.inf,
            "max": -np.inf
        }
    }

    # Script detection
    script_patterns = {
        "Latin": r"[A-Za-z\u00C0-\u02AF]",
        "Devanagari": r"[\u0900-\u097F]",
        "Bengali": r"[\u0980-\u09FF]",
        "Gurmukhi": r"[\u0A00-\u0A7F]",
        "Gujarati": r"[\u0A80-\u0AFF]",
        "Odia": r"[\u0B00-\u0B7F]",
        "Tamil": r"[\u0B80-\u0BFF]",
        "Telugu": r"[\u0C00-\u0C7F]",
        "Kannada": r"[\u0C80-\u0CFF]",
        "Malayalam": r"[\u0D00-\u0D7F]"
    }

    script_counts = {
        "business_name": {s: 0 for s in script_patterns},
        "business_address": {s: 0 for s in script_patterns}
    }

    # Reservoir containing random-priority rows
    reservoir = pd.DataFrame()

    for chunk in pd.read_csv(
        path,
        sep="\t",
        chunksize=chunksize
    ):

        n = len(chunk)
        total_rows += n

        # -------------------------
        # Missing values
        # -------------------------
        for col in missing:
            missing[col] += chunk[col].isna().sum()

        # -------------------------
        # Country counts
        # -------------------------
        counts = chunk["country"].value_counts(dropna=False)

        for country, count in counts.items():
            country = str(country)
            country_counts[country] = (
                country_counts.get(country, 0) + int(count)
            )

        # -------------------------
        # String length statistics
        # -------------------------
        for column, key in [
            ("business_name", "name"),
            ("business_address", "address")
        ]:

            lengths = chunk[column].fillna("").astype(str).str.len()

            stats[key]["count"] += len(lengths)
            stats[key]["sum"] += lengths.sum()
            stats[key]["sum_sq"] += np.square(lengths).sum()
            stats[key]["min"] = min(
                stats[key]["min"], lengths.min()
            )
            stats[key]["max"] = max(
                stats[key]["max"], lengths.max()
            )

        # -------------------------
        # Script detection
        # -------------------------
        for column in ["business_name", "business_address"]:

            text = chunk[column].fillna("").astype(str)

            for script, pattern in script_patterns.items():

                count = text.str.contains(
                    pattern,
                    regex=True,
                    na=False
                ).sum()

                script_counts[column][script] += int(count)

        # -------------------------
        # Random sample
        # -------------------------

        chunk = chunk.copy()

        # Every row gets a random priority
        chunk["_random_priority"] = rng.random(n)

        if len(reservoir) == 0:
            reservoir = chunk
        else:
            reservoir = pd.concat(
                [reservoir, chunk],
                ignore_index=True
            )

        # Keep only the rows with the smallest random priorities
        if len(reservoir) > sample_size:
            reservoir = reservoir.nsmallest(
                sample_size,
                "_random_priority"
            )

    reservoir = (
        reservoir
        .drop(columns="_random_priority")
        .reset_index(drop=True)
    )

    # -------------------------
    # Final statistics
    # -------------------------

    final_stats = {}

    for key in ["name", "address"]:

        s = stats[key]

        mean = s["sum"] / s["count"]

        variance = (
            s["sum_sq"] / s["count"]
            - mean ** 2
        )

        final_stats[key] = {
            "mean": mean,
            "std": np.sqrt(max(variance, 0)),
            "min": s["min"],
            "max": s["max"]
        }

    return {
        "rows": total_rows,
        "missing": missing,
        "countries": country_counts,
        "length_stats": final_stats,
        "script_counts": script_counts,
        "sample": reservoir
    }

In [6]:
print("Scanning Source 1...")

s1_results = scan_source_file(
    train_s1_path,
    sample_size=SAMPLE_SIZE,
    chunksize=CHUNK_SIZE,
    seed=RANDOM_SEED
)

print("Done.")
print("Total rows:", s1_results["rows"])

Scanning Source 1...
Done.
Total rows: 2206821


In [7]:
display(s1_results["sample"].head())

,entity_id,business_name,business_address,country
0,S1-311426597,Informatics Sangh,"Centre Chsl Cross Rd No9, Maharashtra, Mumbai,...",India
1,S1-132046045,Shree Hospitality Private Limited,"32Nd Floor, Sunshine Tower, Senapati Bapat Mar...",India
2,S1-209096536,American Trust,"4018 Abasco Street, Fayetteville, NC",US
3,S1-346269526,"Drain, Partain & Green LP","NC, Plymouth, 383 64, Unit 5",US
4,S1-986351495,Vision Care,"1606 A Eugene Avenue, High Point, NC",US


In [8]:
print("Missing:")
print(s1_results["missing"])

print("\nCountries:")
print(s1_results["countries"])

print("\nName length:")
print(s1_results["length_stats"]["name"])

print("\nAddress length:")
print(s1_results["length_stats"]["address"])

print("\nName scripts:")
print(s1_results["script_counts"]["business_name"])

print("\nAddress scripts:")
print(s1_results["script_counts"]["business_address"])

Missing:
{'entity_id': np.int64(0), 'business_name': np.int64(0), 'business_address': np.int64(0), 'country': np.int64(0)}

Countries:
{'US': 1323633, 'India': 883188}

Name length:
{'mean': np.float64(24.03440333402664), 'std': np.float64(7.740530878949385), 'min': 3, 'max': 105}

Address length:
{'mean': np.float64(52.06621289175697), 'std': np.float64(25.329718602877556), 'min': 11, 'max': 256}

Name scripts:
{'Latin': 2206821, 'Devanagari': 0, 'Bengali': 0, 'Gurmukhi': 0, 'Gujarati': 0, 'Odia': 0, 'Tamil': 0, 'Telugu': 0, 'Kannada': 0, 'Malayalam': 0}

Address scripts:
{'Latin': 2206821, 'Devanagari': 0, 'Bengali': 0, 'Gurmukhi': 0, 'Gujarati': 0, 'Odia': 0, 'Tamil': 0, 'Telugu': 0, 'Kannada': 0, 'Malayalam': 0}


In [9]:
print("Scanning Source 2...")

s2_results = scan_source_file(
    train_s2_path,
    sample_size=SAMPLE_SIZE,
    chunksize=CHUNK_SIZE,
    seed=RANDOM_SEED
)

print("Done.")

print("Scanning Source 3...")

s3_results = scan_source_file(
    train_s3_path,
    sample_size=SAMPLE_SIZE,
    chunksize=CHUNK_SIZE,
    seed=RANDOM_SEED
)

print("Done.")

Scanning Source 2...
Done.
Scanning Source 3...
Done.


In [10]:
for name, result in [
    ("S1", s1_results),
    ("S2", s2_results),
    ("S3", s3_results)
]:
    print(f"\n========== {name} ==========")

    print("Rows:", result["rows"])
    print("Missing:", result["missing"])
    print("Countries:", result["countries"])

    print("\nName lengths:")
    print(result["length_stats"]["name"])

    print("\nAddress lengths:")
    print(result["length_stats"]["address"])


========== S1 ==========
Rows: 2206821
Missing: {'entity_id': np.int64(0), 'business_name': np.int64(0), 'business_address': np.int64(0), 'country': np.int64(0)}
Countries: {'US': 1323633, 'India': 883188}

Name lengths:
{'mean': np.float64(24.03440333402664), 'std': np.float64(7.740530878949385), 'min': 3, 'max': 105}

Address lengths:
{'mean': np.float64(52.06621289175697), 'std': np.float64(25.329718602877556), 'min': 11, 'max': 256}

========== S2 ==========
Rows: 5034616
Missing: {'entity_id': np.int64(0), 'business_name': np.int64(2), 'business_address': np.int64(168967), 'country': np.int64(0)}
Countries: {'US': 3016817, 'India': 2017799}

Name lengths:
{'mean': np.float64(25.103574135544797), 'std': np.float64(8.89469358591325), 'min': 0, 'max': 104}

Address lengths:
{'mean': np.float64(46.22590858965212), 'std': np.float64(24.84466567430024), 'min': 0, 'max': 249}

========== S3 ==========
Rows: 5285603
Missing: {'entity_id': np.int64(0), 'business_name': np.int64(13), 'busi

In [11]:
for name, result in [
    ("S1", s1_results),
    ("S2", s2_results),
    ("S3", s3_results)
]:

    print(f"\n========== {name} SCRIPTS ==========")

    print("\nBusiness name:")
    print(
        pd.Series(
            result["script_counts"]["business_name"]
        ).sort_values(ascending=False)
    )

    print("\nBusiness address:")
    print(
        pd.Series(
            result["script_counts"]["business_address"]
        ).sort_values(ascending=False)
    )


========== S1 SCRIPTS ==========

Business name:
Latin         2206821
Devanagari          0
Bengali             0
Gurmukhi            0
Gujarati            0
Odia                0
Tamil               0
Telugu              0
Kannada             0
Malayalam           0
dtype: int64

Business address:
Latin         2206821
Devanagari          0
Bengali             0
Gurmukhi            0
Gujarati            0
Odia                0
Tamil               0
Telugu              0
Kannada             0
Malayalam           0
dtype: int64

========== S2 SCRIPTS ==========

Business name:
Latin         4578661
Devanagari     269424
Telugu          39323
Kannada         37211
Tamil           33781
Gujarati        30929
Bengali         30723
Malayalam       18773
Odia             7493
Gurmukhi         6688
dtype: int64

Business address:
Latin         4865649
Devanagari     277523
Kannada         38560
Tamil           34777
Telugu          34382
Bengali         31770
Gujarati        31543
Malayalam

In [13]:
ground_truth_sample = random_sample_tsv(
    ground_truth_path,
    sample_size=1000,
    chunksize=CHUNK_SIZE,
    seed=RANDOM_SEED
)

print(ground_truth_sample.shape)
display(ground_truth_sample.head(10))

(1000, 2)


,source1_entity_id,matched_entity_ids
0,S1-263248434,"S2-430851093,S3-151896956"
1,S1-56988120,"S2-253993385,S2-22799521,S3-647000747,S3-43834..."
2,S1-189992826,"S2-668584908,S3-724148110,S3-803143058,S3-8197..."
3,S1-346380440,"S2-673267740,S3-628205708,S3-605616502,S3-5686..."
4,S1-209355677,"S2-775913612,S2-711291721,S2-996187330,S2-9207..."
5,S1-229363644,"S3-617944222,S3-691291875,S3-908098186,S3-4614..."
6,S1-671914501,"S2-721797377,S2-865696249,S3-369715343,S3-3225..."
7,S1-738273577,"S2-254799132,S2-444574535,S3-855764529"
8,S1-57499547,"S2-928330624,S2-516997259,S3-23740322,S3-39680..."
9,S1-821333495,"S2-412864869,S2-252278951,S3-271399559"


In [14]:
import re
import unicodedata

def normalize_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text)

    # Normalize Unicode representation
    text = unicodedata.normalize("NFKC", text)

    # Case-insensitive normalization
    text = text.casefold()

    # Replace punctuation/symbols with spaces
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [15]:
for df in [train_s1, train_s2, train_s3]:
    df["business_name_clean"] = df["business_name"].apply(normalize_text)
    df["business_address_clean"] = df["business_address"].apply(normalize_text)

NameError: name 'train_s1' is not defined